# Long-context probe: `max_new_tokens` comparison

This notebook runs the 5 SQuAD-v2 questions on the **longest filtered contexts**
from `LeninGF/autotrain-data-robberyclassification` and compares
`max_new_tokens` in **256, 512 and 1024**.

It is self-contained: it loads the source dataset, applies the same
82–150 word filter used by `scripts/build_dataset_local_gpu.py`, picks the
longest contexts, and calls the local model directly with
`outlines`-constrained JSON generation.

Run it from the repository root.

In [1]:
import json
import os
import sys
import time

sys.path.insert(0, os.path.join(os.getcwd(), "scripts"))

from transformers import AutoTokenizer
from datasets import load_dataset

from local_qa_generation import (
    MODEL_REGISTRY,
    PREGUNTAS_COMUNES,
    AnswerSchema,
    build_prompt,
    load_local_model,
)

# Config -------------------------------------------------------------------
DATASET_PATH = "LeninGF/autotrain-data-robberyclassification"
MODEL_KEY = "qwen2.5-3b-instruct"  # or "gemma-3-1b-it"
GPU_ID = 0
MIN_WORDS = 82
MAX_WORDS = 150
NUM_CONTEXTS = 3                 # number of longest contexts to probe
MAX_NEW_TOKENS_VALUES = [256, 512, 1024]
OUTPUT_RESULTS = "dataset/long_context_probe_results.jsonl"

/var/lib/datausers_jupyterhub/lfalconi_storage/miniforge3/envs/pyt-eqa-fge/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
print("Loading source dataset ...")
ds = load_dataset(DATASET_PATH)
train = ds["train"]

filtered = []
for i, sample in enumerate(train):
    text = sample["relato"]
    wc = len(text.split())
    if MIN_WORDS <= wc <= MAX_WORDS:
        filtered.append((i, text, wc))

filtered.sort(key=lambda r: len(r[1]), reverse=True)
longest = filtered[:NUM_CONTEXTS]
print(f"Filtered contexts: {len(filtered)}")
for idx, text, wc in longest:
    print(f"  idx={idx} words={wc} chars={len(text)}")

Loading source dataset ...
Filtered contexts: 174594
  idx=319552 words=132 chars=1055
  idx=165321 words=150 chars=1025
  idx=138955 words=150 chars=1010


In [3]:
# Optional HF login (needed for gated models like Gemma; harmless for Qwen).
try:
    from huggingface_hub import login
    env_path = ".env"
    if os.path.exists(env_path):
        with open(env_path, encoding="utf-8") as f:
            for line in f:
                if line.startswith("HUGGINGFACE_TOKEN="):
                    login(line.strip().split("=", 1)[1])
                    break
except Exception as e:
    print("HF login skipped:", e)

print("Loading model ...")
model = load_local_model(MODEL_KEY, gpu_ids=[GPU_ID], quantize_4bit=True)
tok = AutoTokenizer.from_pretrained(MODEL_REGISTRY[MODEL_KEY]["hf_name"])
print("Model loaded.")

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


HF login skipped: Invalid user token.
Loading model ...
Loading Qwen/Qwen2.5-3B-Instruct on logical gpu ids [0] ...


Loading weights: 100%|██████████| 434/434 [00:02<00:00, 186.21it/s]


Model loaded.


In [4]:
def run_probe(context, question, max_new_tokens):
    """Run one generation and return a result dict."""
    prompt = build_prompt(context, question)
    started = time.time()
    result = {
        "context_id": None,
        "question": question,
        "max_new_tokens": max_new_tokens,
        "ok": False,
        "error": None,
        "answer_text": None,
        "answer_chars": None,
        "answer_tokens": None,
        "found_in_context": None,
        "elapsed_s": None,
    }
    try:
        raw = model(prompt, AnswerSchema, max_new_tokens=max_new_tokens, do_sample=False)
        parsed = AnswerSchema.model_validate_json(raw).model_dump()
        answer = parsed["answer_text"]
        result.update(
            ok=True,
            answer_text=answer,
            answer_chars=len(answer),
            answer_tokens=len(tok.encode(answer, add_special_tokens=False)),
            found_in_context=answer in context,
        )
    except Exception as e:
        result["error"] = f"{type(e).__name__}: {str(e)[:200]}"
    result["elapsed_s"] = round(time.time() - started, 2)
    return result

In [5]:
results = []
for ctx_idx, text, wc in longest:
    for question in PREGUNTAS_COMUNES:
        for mnt in MAX_NEW_TOKENS_VALUES:
            r = run_probe(text, question, mnt)
            r["context_idx"] = ctx_idx
            r["context_chars"] = len(text)
            r["context_words"] = wc
            results.append(r)
            print(
                f"ctx={ctx_idx} q='{question[:35]}...' max_tokens={mnt} "
                f"ok={r['ok']} chars={r['answer_chars']} tokens={r['answer_tokens']} "
                f"err={r['error']}"
            )

ctx=319552 q='¿Qué objetos fueron robados?...' max_tokens=256 ok=True chars=31 tokens=9 err=None
ctx=319552 q='¿Qué objetos fueron robados?...' max_tokens=512 ok=True chars=31 tokens=9 err=None
ctx=319552 q='¿Qué objetos fueron robados?...' max_tokens=1024 ok=True chars=31 tokens=9 err=None
ctx=319552 q='¿En qué fecha ocurrió el incidente?...' max_tokens=256 ok=True chars=28 tokens=10 err=None
ctx=319552 q='¿En qué fecha ocurrió el incidente?...' max_tokens=512 ok=True chars=28 tokens=10 err=None
ctx=319552 q='¿En qué fecha ocurrió el incidente?...' max_tokens=1024 ok=True chars=28 tokens=10 err=None
ctx=319552 q='¿A qué hora sucedió el robo?...' max_tokens=256 ok=True chars=9 tokens=7 err=None
ctx=319552 q='¿A qué hora sucedió el robo?...' max_tokens=512 ok=True chars=9 tokens=7 err=None
ctx=319552 q='¿A qué hora sucedió el robo?...' max_tokens=1024 ok=True chars=9 tokens=7 err=None
ctx=319552 q='¿En qué dirección o entre qué calle...' max_tokens=256 ok=True chars=35 tokens=11 err=Non

In [6]:
os.makedirs(os.path.dirname(OUTPUT_RESULTS), exist_ok=True)
with open(OUTPUT_RESULTS, "w", encoding="utf-8") as f:
    for r in results:
        f.write(json.dumps(r, ensure_ascii=False) + "\n")
print(f"Saved {len(results)} rows to {OUTPUT_RESULTS}")

# Quick summary table
from collections import defaultdict
summary = defaultdict(lambda: {"ok": 0, "max_chars": 0, "max_tokens": 0, "errors": 0, "total": 0})
for r in results:
    s = summary[r["max_new_tokens"]]
    s["total"] += 1
    if r["ok"]:
        s["ok"] += 1
        s["max_chars"] = max(s["max_chars"], r["answer_chars"])
        s["max_tokens"] = max(s["max_tokens"], r["answer_tokens"])
    else:
        s["errors"] += 1

print("\nSummary by max_new_tokens:")
for mnt in MAX_NEW_TOKENS_VALUES:
    s = summary[mnt]
    print(
        f"  max_new_tokens={mnt}: ok={s['ok']}/{s['total']} errors={s['errors']} "
        f"max_answer_chars={s['max_chars']} max_answer_tokens={s['max_tokens']}"
    )

Saved 45 rows to dataset/long_context_probe_results.jsonl

Summary by max_new_tokens:
  max_new_tokens=256: ok=14/15 errors=1 max_answer_chars=43 max_answer_tokens=14
  max_new_tokens=512: ok=15/15 errors=0 max_answer_chars=551 max_answer_tokens=252
  max_new_tokens=1024: ok=15/15 errors=0 max_answer_chars=551 max_answer_tokens=252


## How to interpret

- If 256 produces `ok=True` for all probes and no `Invalid JSON: EOF` errors,
  it is enough for the longest contexts in this dataset.
- If 256 still truncates but 512 does not, use 512 as the default for the build.
- 1024 is only needed if answers approach the full context length (~588 tokens
  for the longest context with the Qwen tokenizer).

To probe the other model, change `MODEL_KEY` to `"gemma-3-1b-it"` and re-run
the model-loading and probe cells.